# 🚀 MangaTranslator - Inpainting Server trên Google Colab (Miễn phí T4 GPU)

Cell này sẽ tự động:
1. Cài đặt các thư viện cần thiết (`fastapi`, `uvicorn`, `pycloudflared`, `diffusers`, `accelerate`).
2. Khởi tạo FastAPI Inpainting Server (chạy trên GPU Tesla T4).
3. Tạo đường hầm Cloudflare Tunnel (`https://xxxx.trycloudflare.com`) hoàn toàn miễn phí, **không cần token ngrok**, **không cần đăng ký tài khoản**.
4. In ra đường link công khai để bạn copy dán vào MangaTranslator UI.

In [ ]:
#@title ▶️ BẤM NÚT NÀY ĐỂ KHỞI CHẠY SERVER (1 Cell duy nhất)
import os
import sys
import time
import subprocess

print("⏳ Đang cài đặt các thư viện cần thiết...")
!pip install -q fastapi uvicorn pycloudflared python-multipart diffusers transformers accelerate

# Kiểm tra GPU
import torch
if not torch.cuda.is_available():
    print("⚠️ CẢNH BÁO: Bạn chưa bật GPU trên Colab! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU rồi chạy lại cell này.")
else:
    print(f"✅ Đã nhận diện GPU: {torch.cuda.get_device_name(0)}")

# Tạo file server.py
server_code = '''import io
import torch
from PIL import Image
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import Response, JSONResponse
from diffusers import StableDiffusionInpaintPipeline, AutopipelineForInpainting

app = FastAPI(title="MangaTranslator Inpainting Server")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
current_pipeline = None
current_model_name = None

def get_pipeline(model_name: str):
    global current_pipeline, current_model_name
    if current_pipeline is not None and current_model_name == model_name:
        return current_pipeline

    print(f"[Colab Server] Đang tải mô hình: {model_name}...")
    if current_pipeline is not None:
        del current_pipeline
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Mặc định dùng Stable Diffusion 1.5 Inpainting (nhẹ, nhanh ~1-2s trên T4)
    repo_id = "runwayml/stable-diffusion-inpainting"
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        repo_id,
        torch_dtype=dtype,
        safety_checker=None,
    ).to(device)

    # Tối ưu bộ nhớ
    if hasattr(pipe, "enable_attention_slicing"):
        pipe.enable_attention_slicing()

    current_pipeline = pipe
    current_model_name = model_name
    print(f"[Colab Server] Đã nạp thành công mô hình {model_name}!")
    return current_pipeline

@app.get("/")
@app.get("/health")
def health_check():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram_gb = round(torch.cuda.memory_allocated(0) / (1024**3), 2) if torch.cuda.is_available() else 0
    return {
        "status": "ok",
        "gpu": gpu_name,
        "loaded_model": current_model_name or "None (Chưa tải)",
        "vram_used_gb": vram_gb
    }

@app.post("/inpaint")
async def inpaint_endpoint(
    image: UploadFile = File(...),
    mask: UploadFile = File(...),
    prompt: str = Form("clean background, remove text"),
    num_inference_steps: int = Form(20),
    guidance_scale: float = Form(7.5),
    seed: int = Form(None),
    model_name: str = Form("sd_inpainting")
):
    try:
        img_bytes = await image.read()
        mask_bytes = await mask.read()

        img_pil = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        mask_pil = Image.open(io.BytesIO(mask_bytes)).convert("L")

        orig_w, orig_h = img_pil.size
        pipe = get_pipeline(model_name)

        # Đảm bảo kích thước chia hết cho 8 cho SD 1.5
        target_w = ((orig_w + 7) // 8) * 8
        target_h = ((orig_h + 7) // 8) * 8
        if (target_w, target_h) != (orig_w, orig_h):
            img_in = img_pil.resize((target_w, target_h), Image.Resampling.LANCZOS)
            mask_in = mask_pil.resize((target_w, target_h), Image.Resampling.NEAREST)
        else:
            img_in = img_pil
            mask_in = mask_pil

        generator = None
        if seed is not None and seed >= 0:
            generator = torch.Generator(device=device).manual_seed(int(seed))

        # Chạy inference
        result = pipe(
            prompt=prompt,
            image=img_in,
            mask_image=mask_in,
            num_inference_steps=max(1, min(50, num_inference_steps)),
            guidance_scale=float(guidance_scale),
            generator=generator,
        ).images[0]

        if (target_w, target_h) != (orig_w, orig_h):
            result = result.resize((orig_w, orig_h), Image.Resampling.LANCZOS)

        out_buf = io.BytesIO()
        result.save(out_buf, format="PNG")
        return Response(content=out_buf.getvalue(), media_type="image/png")
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})
'''

with open("server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

# Khởi động FastAPI server ở cổng 8000 nền
print("🚀 Đang khởi động FastAPI Server...")
server_proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(3)

# Mở tunnel Cloudflare bằng pycloudflared
print("🌐 Đang tạo link Public Tunnel qua Cloudflare...")
from pycloudflared import try_cloudflare
tunnel_url = try_cloudflare(port=8000).tunnel

print("\n" + "="*60)
print("🎉 SERVER COLAB ĐÃ SẴN SÀNG!")
print(f"🔗 URL CỦA BẠN: {tunnel_url}")
print("\n👉 HƯỚNG DẪN:")
print(f"1. Copy đường link trên: {tunnel_url}")
print("2. Mở MangaTranslator -> Phần Outside Text Settings:")
print("   - Chọn Flux Backend: 'Remote Colab (T4 GPU)'")
print("   - Dán link vào ô 'Colab Server URL'")
print("   - Bấm nút 'Kiểm tra kết nối Colab' để xác nhận!")
print("="*60 + "\n")

# Giữ cell luôn chạy
try:
    server_proc.wait()
except KeyboardInterrupt:
    server_proc.terminate()
    print("Server đã dừng.")
